In [2]:
# load data 

import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "/Users/youting/hgcal_energy_analysis/hgcal_dataset.npz"
d = np.load(DATA_PATH)

measuredE = d["measuredE"]   # (N, 34)
absorberE = d["absorberE"]
totalE    = d["totalE"]
voldEdx   = d["voldEdx"]
beam_E    = d["beam_energy"]  # GeV
run_num   = d["run_number"]

for k in d.files:
    print(f"{k:12s} {d[k].shape} {d[k].dtype}")

measuredE    (307424, 34) float64
absorberE    (307424, 34) float64
totalE       (307424, 34) float64
voldEdx      (307424, 34) float64
event_id     (307424,) uint32
beam_energy  (307424,) int64
run_number   (307424,) int64
source_file  (307424,) <U52


In [3]:
# linear regression
# train & test?

EMPTY_IDX  = [1, 5, 7, 11, 13, 17, 19, 23, 25, 28, 30]
ACTIVE_IDX = [i for i in range(34) if i not in EMPTY_IDX]

X = d["measuredE"][:,ACTIVE_IDX] #input
y = d["beam_energy"] #goal

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y values: {np.unique(y)}")



X shape: (307424, 23)
y shape: (307424,)
y values: [ 20  50  80 100 150 200 300]


In [4]:
# split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

print(f"train(X, y): ({X_train.shape},{y_train.shape})")
print(f"test(X, y): ({X_test.shape},{y_test.shape})")

for E in np.unique(y):
    n_tr = (y_train == E).sum()
    n_te = (y_test == E).sum()

    print(f"{E:.4f}GeV, train:{n_tr:6d}, test:{n_te:6d}, testing ratio:{(n_te)/(n_tr + n_te):.4f}")
    

train(X, y): ((245939, 23),(245939,))
test(X, y): ((61485, 23),(61485,))
20.0000GeV, train: 40000, test: 10000, testing ratio:0.2000
50.0000GeV, train: 40000, test: 10000, testing ratio:0.2000
80.0000GeV, train: 40000, test: 10000, testing ratio:0.2000
100.0000GeV, train: 40000, test: 10000, testing ratio:0.2000
150.0000GeV, train:  8000, test:  2000, testing ratio:0.2000
200.0000GeV, train: 40000, test: 10000, testing ratio:0.2000
300.0000GeV, train: 37939, test:  9485, testing ratio:0.2000


In [5]:
# fitting
from sklearn.linear_model import LinearRegression

keep = np.abs(X_train).sum(axis=0) > 0
kept_sections = np.array(ACTIVE_IDX)[keep]

X_train_k = X_train[:,keep]
X_test_k = X_test[:,keep]

print("kept_sections:", kept_sections)

model = LinearRegression()
model.fit(X_train_k, y_train)

print("interception b:", model.intercept_)
for sec, w in zip(kept_sections, model.coef_):
    print(f"section:{sec:2d}, weight:{w:.5f}")
print("train R square:", model.score(X_train_k, y_train))


kept_sections: [ 2  4  8 10 14 16 20 22 26 31]
interception b: 0.2833147920913177
section: 2, weight:0.16751
section: 4, weight:0.22642
section: 8, weight:0.21113
section:10, weight:0.21871
section:14, weight:0.21136
section:16, weight:0.20731
section:20, weight:0.20900
section:22, weight:0.22109
section:26, weight:0.31883
section:31, weight:0.65466
train R square: 0.9983717658180336


/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [6]:
# Check warning
print("X_train_k 全為有限數:", np.isfinite(X_train_k).all())
print("X_train_k 最大值:", X_train_k.max())

pred_manual = (X_train_k * model.coef_).sum(axis=1) + model.intercept_
print("預測值全為有限數:", np.isfinite(pred_manual).all())

ss_res = ((y_train - pred_manual) ** 2).sum()
ss_tot = ((y_train - y_train.mean()) ** 2).sum()
print("手算 R²:", 1 - ss_res / ss_tot)

X_train_k 全為有限數: True
X_train_k 最大值: 340.08569611356245
預測值全為有限數: True
手算 R²: 0.9983717658180336


In [9]:
# predict
#y_pred = (X_test_k * model.coef_).sum(axis=1)+model.intercept_
model.predict(X_test_k)

print("y_pred shape:", y_pred.shape)
print("ratio of finite value:", np.isfinite(y_pred).mean())

y_pred shape: (61485,)
ratio of finite value: 1.0


/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/envs/calo/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [13]:
# Evaluation
print(f"{'E_true':>7s} {'mean_pred':>10s} {'std_pred':>9s} {'resolution':>11s} {'bias':>8s}")

for E in np.unique(y_test):
    m = y_test == E
    mu = y_pred[m].mean()
    sig = y_pred[m].std()
    print(f"{E:7d} {mu:10.3f} {sig:9.3f} {sig/mu:11.4f} {(mu-E)/E:8.4f}")

 E_true  mean_pred  std_pred  resolution     bias
     20     20.225     1.379      0.0682   0.0113
     50     50.155     2.217      0.0442   0.0031
     80     80.147     2.899      0.0362   0.0018
    100    100.164     3.242      0.0324   0.0016
    150    147.263     4.039      0.0274  -0.0182
    200    200.055     4.750      0.0237   0.0003
    300    299.757     6.091      0.0203  -0.0008
